    # ELEC3901 Robotic Manipulators — Lab 1: Environment set-up and meeting the OMX

    **Week 1: Introduction to Robotic Manipulators**  |  Reading: RVC3 Ch. 1; Ch. 2.1 (2D pose); Ch. 7 introduction  |  Learning outcomes: C1, C12, C13

    ## Plan for this session (2 h)
    1. Safety briefing and sign-off: power isolation, pinch points, workspace clearance, torque-off protocol, emergency stop
2. Check the environment set up in the pre-lab (week00 of your LABS repository); record the arm ID; run the per-session camera calibration from the mat (under a minute; repeated at the start of every lab, because you may not be on the same arm or camera as last week)
3. Connect to the OMX-F via its OpenRB-150 (USB-C); read joint positions; torque-off 'teach' mode; identify positive joint directions and limits by hand
4. Command a single joint in small increments with profile velocity limits; log and plot joint position over time
5. 2D pose exercises with spatialmath SE2: compose, invert, plot frames; relate to a top-down view of the OMX base

    **Checkpoint for sign-off:** Demonstrator sign-off: environment working, arm connected, joint positions read and plotted

    **Coding tasks.** Three cells in this notebook are marked *Task* and contain only a function signature and a `TODO`. You write the body; the
    self-check cell below each one compares your result with a hidden reference value and prints OK or x. Later parts of the lab use *your* functions.
    Reference versions of previous weeks' tasks are imported from `omx_ref.py` so last week never blocks this week; if a task defeats you today, ask a
    demonstrator rather than lose the hardware time — but the functions you write here are the ones your assignment will be built from.

    > **Safety reminder.** Keep the workspace clear of hands and cables before enabling torque. Always set a conservative
    > profile velocity (`arm.set_profile(60, 20)` or lower) before commanding motion. If anything unexpected happens, press the
    > power isolator on the bench first, then think. Torque-off (`arm.torque(False)`) before touching the arm.

## Part A — Environment check
Run the cell below. If any import fails, run `python week00/check_setup.py` from the repository folder (the pre-lab): it names the
fix. Make sure the notebook's kernel (top right in VS Code) is **elec3901**. Still stuck: ask a demonstrator.

In [ ]:
import roboticstoolbox as rtb, spatialmath, numpy as np
print('roboticstoolbox', rtb.__version__)
print('spatialmath', spatialmath.__version__)
try:
    import dynamixel_sdk; print('dynamixel_sdk OK')
except ImportError:
    print('dynamixel_sdk MISSING -> pip install dynamixel-sdk')
import cv2; print('opencv', cv2.__version__, 'aruco', hasattr(cv2, 'aruco'))
# cameras by NAME: camera number 0 is usually the laptop's own webcam, not the wrist camera
try:
    from pygrabber.dshow_graph import FilterGraph                   # Windows
    cams = [n.strip() for n in FilterGraph().get_input_devices()]
except ImportError:
    import glob                                                      # Linux
    cams = [open(p).read().strip() for p in sorted(glob.glob('/sys/class/video4linux/video*/name'))]
print('cameras:', cams)
print('wrist camera (Innomaker) connected:', any('innomaker' in c.lower() for c in cams))
# every lab starts with the same cells: set-up (imports), connect (the arm) and camera (calibrates it from the mat;
# re-run that cell alone to repeat it). You calibrate EVERY lab: you may not be on the same arm (or camera) as last
# week, and the manual focus may have been knocked. In this first lab the camera cell comes after Part B, once you
# have checked the joints and moved one yourself - it moves the arm on its own.

In [ ]:
import sys, os
# Repository layout: weekNN/ (notebooks), scripts/ (helper modules), data/ (your measurements, shared by every week).
for _up in ('.', '..', os.path.join('..', '..')):
    if os.path.isdir(os.path.join(_up, 'scripts')):
        _root = os.path.abspath(_up); sys.path.insert(0, os.path.join(_root, 'scripts'))
        os.makedirs(os.path.join(_root, 'data'), exist_ok=True); os.chdir(os.path.join(_root, 'data'))   # files save/load here
        break
import numpy as np, matplotlib.pyplot as plt, time, cv2
import roboticstoolbox as rtb
from spatialmath import SE3, SO3, SE2, UnitQuaternion
from spatialmath.base import trplot, tranimate
from omx_model import omx_robot, omx_ik, Q_LIM, T_EC, P_TIP_E, mount_tilt, look_down_pitch
from omx import OMX
from omx_camera import Camera, tip_in_mat, predicted_T_CM, pose_error
from omx_calib import small_T
from labcheck import check
NOM = dict(d1=0.0975, l2z=0.11315, l2x=0.0415, l3=0.162, l4=0.123)   # nominal link parameters (m)

np.set_printoptions(precision=4, suppress=True)
%matplotlib inline
omx = omx_robot()              # Robotics Toolbox model (nominal)
T_EC_CAD = T_EC   # CAD camera mount: a PLACEHOLDER (it has the camera looking along the tool). Real mounts are tilted ~60 deg; Lab 2
                  # measures yours, and from Lab 3 on the MOUNT cell re-measures it every session and replaces T_EC
HOVER = 0.09      # tip hover height with the tool vertical: the OMX-F reaches r ~ 0.24 m there (touching the mat: r ~ 0.28 m)
CAM_HOVER = 0.15  # tip height for camera measurements, tool pitched with look_down_pitch(T_EC) so the camera looks straight down:
                  # the camera is then ~180 mm up, enough to see the marker ring around the clear zone

In [ ]:
# --- connect to the arm --------------------------------------------------------------------------------------------
# Safe to re-run: an existing connection is closed first, so the COM port is released before it is opened again.
# SIMULATE = True runs the MuJoCo digital twin (omx_mujoco.py) instead of the real arm: same API, a rendered
# wrist camera, and optional hidden model errors so you can test calibration against ground truth.
SIMULATE = False
PORT = 'auto'                  # finds the OpenRB-150 by its USB id; or set it, e.g. 'COM4' (Device Manager) / '/dev/ttyACM0'
# --- which arm are you on today? Read the asset label on the base. Everything you measure is tagged with it,
#     because zero offsets, camera intrinsics and the camera mount are all properties of THIS arm.
ARM_ID = 'OMX-00'            # EDIT every session
if 'arm' in globals():
    try:
        arm.close()
    except Exception as e:
        print('closing the previous connection:', e)
    del arm
if SIMULATE:
    from omx_mujoco import OMXSim, SimCamera
    arm = OMXSim(view=True)    # view=False if no display; set kin_error=..., joint_offsets=... to hide errors
    # Video: arm.record('run.mp4') writes the wrist camera and a scene camera side by side (same size) while the arm
    # moves; arm.stop_recording() when done.
else:
    arm = OMX(PORT)
print('connected:', 'digital twin' if SIMULATE else arm.port_name, '| arm', ARM_ID)

## Part B — Meet the OMX (and your first coding task)
### B1. Read the joint state
> **Support the arm with one hand BEFORE you run the next cell.** It switches torque off, and the arm then drops under gravity.

With torque off the arm is limp. For the next 60 s the cell shows the four joint angles live: move each joint by hand, gently, and fill in
the table - which joint is which ID, which direction is positive, and roughly where the mechanical limits are. Run it again if you need more time.

In [ ]:
from IPython.display import display
arm.torque(False)
readout = display('', display_id=True)
t0 = time.time()
while time.time() - t0 < 60:
    q = np.degrees(arm.read_q())
    msg = f'joints 1-4 (deg): {np.round(q, 1)}      {60 - (time.time() - t0):3.0f} s left'
    readout.update(msg) if readout is not None else print(msg, end='\r')   # live line in Jupyter; plain print elsewhere
    time.sleep(0.2)

| Joint | Dynamixel ID | Positive direction (describe) | Approx. min (deg) | Approx. max (deg) |
|---|---|---|---|---|
| 1 | 11 | | | |
| 2 | 12 | | | |
| 3 | 13 | | | |
| 4 | 14 | | | |

Compare your findings with `Q_LIM` in `omx_model.py`. **If a joint moves the opposite way to the model's convention, tell a demonstrator** — the sign table in `omx.py` may need editing for your arm.

### B2. Move one joint under control
Move the arm by hand to roughly the home pose (base straight ahead, link 2 leaning back about 60 degrees, the forearm folded down about 60
degrees and the gripper pitched about 30 degrees below horizontal, pointing at the mat), then enable torque with a gentle profile.

### Task 1 — Log a joint step (open task)
Write `step_log(arm, joint, targets_deg, samples=50, period=0.02)` that commands joint `joint` (0-3) through the list of targets
in turn, holding the other joints where they are, and returns `(t, q)` — time stamps (s) and the measured angle of that joint (rad)
sampled `samples` times per target. Then call it on joint 4 with `[0, 20, 40, 20, 0, -20, 0]` and plot the result.

**Acceptance:** the plot shows a staircase with a visible rise time on each step; `t` is monotonic; `q` is in radians.
Show a demonstrator.

In [ ]:
def step_log(arm, joint, targets_deg, samples=50, period=0.02):
    """Command joint `joint` through targets_deg; return (t, q) arrays of the measured joint angle."""
    # TODO: write your implementation here
    raise NotImplementedError


_Open task: no automatic check. Meet the acceptance criteria above and show a demonstrator._

In [ ]:
arm.torque(True); arm.set_profile(velocity=40, acceleration=10); arm.home(wait=True)
t, q4 = step_log(arm, 3, [0, 20, 40, 20, 0, -20, 0])
plt.plot(t, np.degrees(q4)); plt.xlabel('t (s)'); plt.ylabel('joint 4 (deg)'); plt.grid(); plt.show()

**Questions (write answers in this cell):**
1. Estimate the time constant of the joint response. What limits it — the servo, or the profile velocity you set?
2. Change `velocity` to 100 and repeat. What changes? Why do we insist on a limit in the teaching lab?
3. What is the smallest position change you can command? (Hint: 4096 ticks per revolution.)

### B3. Calibrate the wrist camera
You now know which way each joint moves. The next cell calibrates the camera for the arm you are on today: with `AUTO = True` the
**arm moves on its own** through 20 views over the mat (about 40 s), with a live camera window. Keep hands, cables and objects
out of the workspace and a hand near the power switch. You will run this cell at the start of every lab from now on.

In [ ]:
# --- wrist camera: manual focus, may have been moved since last week -> re-estimate intrinsics NOW from the mat -------
# AUTO = True: the ARM calibrates the camera itself - it moves through 20 views aimed at the mat and takes a frame at
#   each once the joints and the image are static (omx_autocal.py; about 40 s). Torque goes ON and the arm MOVES:
#   keep hands, cables and objects out of the workspace. A live window shows each view.
# AUTO = False: move the camera over the mat by hand instead (torque off, live window, pause at each view).
# Re-run THIS cell on its own to repeat the calibration: the arm connection above is not touched.
# The gripper is left OPEN for all camera work (closed fingers hide the mat); arm.rest() closes it at the end of the lab.
AUTO = True
CAM_INDEX = 'Innomaker'         # the wrist camera by NAME (a laptop's own webcam is usually camera 0); or a number
if globals().get('cam') is not None:
    try:
        cam.close()                # release the camera held by the previous run
    except Exception:
        pass
cam = None
try:
    if SIMULATE:
        cam = SimCamera(arm); raise StopIteration      # twin: exact intrinsics, no per-session calibration needed
    cam = Camera(CAM_INDEX)
    import os
    import omx_camera
    _kb = [p for p in ('camera_K_baseline.npz', os.path.join(omx_camera.HERE, 'camera_K_baseline.npz')) if os.path.exists(p)]
    K0 = np.load(_kb[0])['K'] if _kb else None   # shared staff baseline (scripts/camera_K_baseline.npz): a starting guess only
    if AUTO:
        from omx_autocal import auto_calibrate
        print('Automatic calibration: the arm is about to move. Keep the workspace clear.')
        K, dist, rms = auto_calibrate(arm, cam, ARM_ID, K0=K0, show=True, stages=1, show_mount=False)   # stages=2: + planned views
    else:
        arm.torque(False)
        print('Torque is OFF. Move the camera over the mat by hand (up to 60 s), PAUSING at each view: vary its height between ~90 and 180 mm AND tilt the wrist')
        print('by at least +/-30 degrees (40 is better) - nearly vertical views at one height give a focal length 10-20% wrong.')
        # show=True opens a live window (press q to stop early); still_px keeps a frame only while you hold the camera still
        K, dist, rms = cam.calibrate_from_mat(n_frames=25, seconds=60, K0=K0, save_as=f'camera_K_session_{ARM_ID}.npz',
                                              show=True, still_px=1.5)
    if rms > 1.5: print('WARNING: rms > 1.5 px - check focus and that the mat is flat, then re-run this cell.')
except StopIteration:
    print('simulated wrist camera ready (K exact)')
except Exception as e:
    if cam is not None:
        try: cam.close()
        except Exception: pass
    cam = None; print('camera not available:', e)

## Part C — 2D pose (RVC3 Ch. 2.1)
Imagine looking down on the OMX. The base frame {B} sits at the joint-1 axis; a part on the mat has its own frame.

### Task 2 — 2D pose from scratch
Before using `SE2`, build it yourself. Write the three functions below with plain NumPy. Angles in radians, lengths in metres.
Compare your matrices with `SE2(x, y, theta).A` afterwards.

In [ ]:
def rot2(theta):
    """2x2 rotation matrix for angle theta (rad)."""
    # TODO: write your implementation here
    raise NotImplementedError

def se2(x, y, theta):
    """3x3 homogeneous transform: translate (x, y) then rotate theta. Returns a NumPy array."""
    # TODO: write your implementation here
    raise NotImplementedError

def transform_point(T, p):
    """Apply a 3x3 transform T to a 2D point p (length-2 array). Returns a length-2 array."""
    # TODO: write your implementation here
    raise NotImplementedError


In [ ]:
# --- self-check (hidden expected values) ---------------------------------------------------------
check('rot2(30 deg)', lambda: rot2(np.radians(30)), 'd425857384', decimals=4)
check('se2(0.15, -0.10, 30 deg)', lambda: se2(0.15, -0.10, np.radians(30)), 'dad5a043df', decimals=4)
check('point (0.05, 0.02) mapped through se2', lambda: transform_point(se2(0.15, -0.10, np.radians(30)), [0.05, 0.02]), '613ddd1660', decimals=4)
check('inverse: T @ inv(T) is identity', lambda: se2(0.1, 0.2, 0.7) @ np.linalg.inv(se2(0.1, 0.2, 0.7)), '51ad9d3fcd', decimals=6)

Now the same thing with the toolbox's `SE2` class — check that its matrices match yours.

In [ ]:
# a pose is a translation followed by a rotation
T_BM = SE2(0.15, -0.10, np.radians(30))     # mat frame relative to base
print(T_BM)
print('rotation matrix:\n', T_BM.R, '\ntranslation:', T_BM.t)

p_M = np.array([0.05, 0.02])                # a point on the mat, in mat coordinates
p_B = (T_BM * p_M).flatten()                # the same point in base coordinates (SE2 * point returns a 2x1 column)
print('point in base frame:', p_B)

SE2().plot(frame='B', color='k', length=0.1)   # frames drawn 0.1 m long, to the scale of the offset
T_BM.plot(frame='M', color='b', length=0.1)
plt.plot(*p_B, 'ro'); plt.axis('equal'); plt.grid(); plt.show()

In [ ]:
# composition and inversion
T_MP = SE2(0.05, 0.02, 0)                   # a 'part' frame on the mat
T_BP = T_BM * T_MP
print('T_BP =\n', T_BP)
print('T_MB (inverse):\n', T_BM.inv())
# check: T_BM * T_MB should be identity
print((T_BM * T_BM.inv()).A)

Show numerically that `SE2(a) * SE2(b) != SE2(b) * SE2(a)` in general. When *does* the order not matter?

### Task 3 — Pointing the arm's plane at a target
Joint 1 rotates the whole arm about z. Write `azimuth(p)` returning the joint-1 angle (rad, in (-pi, pi]) that puts a
base-frame point `p = [x, y, ...]` in the arm's vertical plane. Then decide what happens for a point *behind* the base.

In [ ]:
def azimuth(p):
    """Joint-1 angle (rad) that points the arm's plane at base-frame point p (x, y[, z])."""
    # TODO: write your implementation here
    raise NotImplementedError


In [ ]:
# --- self-check (hidden expected values) ---------------------------------------------------------
check('azimuth of (0.20, 0.05)', lambda: azimuth([0.20, 0.05, 0.1]), '35513b339e', decimals=5)
check('azimuth of (-0.10, 0.10)', lambda: azimuth([-0.10, 0.10]), '36335da3bf', decimals=5)
check('azimuth of (0.3, -0.3, 0)', lambda: azimuth([0.3, -0.3, 0.0]), '059dc70e0a', decimals=5)

## Before you leave
1. Park the arm (`arm.rest()`: the folded rest pose, where it stays put with torque off), then `arm.torque(False)` and `arm.close()`.
   Never switch torque off at home or mid-task: the arm drops under gravity.
2. Switch off the bench power supply and tidy cables.
3. Show your checkpoint output to a demonstrator for sign-off.
4. Commit this notebook and any data files to your own LABS repository and **push** (Sync Changes): later labs read them. The assessment repository (issued in week 6) starts fresh; copy across any code you want to reuse.

In [ ]:
arm.set_profile(40, 10); arm.rest(wait=True)     # folded rest pose: stable with torque off
arm.torque(False)
arm.close()
if globals().get('cam') is not None: cam.close()